In [ ]:
from langchain_openai import OpenAIEmbeddings
from langchain_community.vectorstores import Chroma
from langchain_core.documents import Document

# -------------------------
# Sample Documents
# -------------------------
documents = [
    Document(page_content="Revenue increased by 30% in Q4", metadata={"topic": "finance"}),
    Document(page_content="Customers prefer subscription pricing", metadata={"topic": "product"}),
    Document(page_content="AI adoption is growing in enterprises", metadata={"topic": "market"})
]

# -------------------------
# Layer 1: Cache (In-Memory)
# -------------------------
cache = {}   # TODO: key = query, value = results


# -------------------------
# Layer 2: Vector Store
# -------------------------
embeddings = OpenAIEmbeddings(model="text-embedding-3-small")

vector_store = Chroma.from_documents(
    documents,
    embedding=embeddings,
    collection_name="multi_tier_demo"
)


# -------------------------
# Layer 3: Raw Fallback
# -------------------------
def fallback_search(query):
    # TODO: simple keyword search
    results = []
    for doc in documents:
        if query.lower() in doc.page_content.lower():
            results.append(doc)
    return results


# -------------------------
# Multi-Tier Retrieval
# -------------------------
def retrieve_context(query):

    # TODO 1: Check cache
    if query in cache:
        print("Cache HIT")
        return cache[query]

    print("Cache MISS → Checking Vector DB")

    # TODO 2: Vector search
    results = vector_store.similarity_search(query, k=2)

    # TODO 3: If no results → fallback
    if not results:
        print("Vector MISS → Falling back")
        results = fallback_search(query)

    # TODO 4: Store in cache
    cache[query] = results

    return results


# -------------------------
# Test Queries
# -------------------------
queries = [
    "pricing strategy",
    "enterprise growth",
    "pricing strategy"   # should hit cache
]

for q in queries:
    print(f"\nQuery: {q}")
    docs = retrieve_context(q)

    for d in docs:
        print("-", d.page_content)